# Synthetic factoid generation

Use **pipeline** for a new end-to-end run: prepare, generate, verify, and finalize. Use **finish** to verify and finalize an existing staged run. Supply question-level prepared `source`, held-out `dev_source`, and `test_dir`; protected dev/test PubMed IDs are excluded.

Only strict verifier-agreed questions are placed in `training_ready/`. Rejected and ineligible records remain under audit/internal artifacts and are never included in SFT or DPO files. SFT exports contain every accepted question. DPO exports conservatively require both distinct questions from a source to pass, then use the other verified relation as a same-evidence hard negative.

`generator_model` and `verifier_model` are independent. `max_new_calls` is a per-API-phase safety cap. For a new 1,090-source full pipeline, start with a suitable explicit budget such as 1,500; cached continuation usually needs a much smaller budget. API phases require both `RUN=True` and `ALLOW_API=True`.

Execution is **off by default** in a clean notebook. Select the project's `.venv` kernel. Each method uses that kernel's Python in a separate worker process.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


## Choose a preset

- **pipeline**: prepare ? generate ? verify ? finalize in one run
- **finish**: verify ? finalize an existing run in one pass
- **prepare**, **generate**, **verify**, **finalize**: individual resumable phases

The configuration below is ready for a new full pipeline and is disabled by default. Set both `RUN=True` and `ALLOW_API=True` when you intentionally start it. To resume, choose the needed preset and set `prepared_run` to the previous `synthetic` directory. Final files are written beneath `synthetic/training_ready/`.

The completed current dataset is in `Artifacts/notebook_runs/synthetic/synthetic_qa/20260928-031536-dcd26860/synthetic/training_ready`.


In [ ]:
CATEGORY = 'synthetic'

DATA = (
    PROJECT_ROOT
    / "data/BioASQ_factoid_sft_prepared"
    / "split_first_train90_dev10_supported_train_seed3407"
)

PRESET = "pipeline"

OVERRIDES = {
    "source": str(DATA / "train_questions.json"),
    "dev_source": str(DATA / "dev_prepared.json"),
    "api_key_file": str(PROJECT_ROOT / "open_ai_api.txt"),

    "source_count": 1090,
    "validation_source_count": 218,
    "max_new_calls": 1500,  # Per-phase cap for a fresh 1,090-source pipeline

    "prepared_run": None,  # Set a prior synthetic directory only when resuming
}

RUN = False
ALLOW_API = False

# Optional sequential comparisons: shared OVERRIDES plus per-variant changes.
VARIANTS = [{"label": "default", "overrides": {}}]

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))


## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [ ]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [ ]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [ ]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])
